# COSMX FOV REPORT FOR SLIDE {{ COSMX_SLIDE_NAME }}
* **Notebook version:** v0.0.2 (memory + speed optimised)
* **Created by:** NIHR Imperial BRC Genomics Facility
* **Maintained by:** NIHR Imperial BRC Genomics Facility
* **Docker image path:** [Dockerfile](https://github.com/imperial-genomics-facility/igf-dockerfiles/tree/main/cosmx/Dockerfile_v1)
* **Notebook code path:** [Templates](https://github.com/imperial-genomics-facility/igf-dockerfiles/tree/main/cosmx/)
* **Created on:** {{ DATE_TAG }}
* **Contact us:** [NIHR Imperial BRC Genomics Facility - Contact us](https://www.imperial.ac.uk/medicine/research-and-impact/facilities/genomics-facility/contact-us/)
* **License:** Apache [License 2.0](https://github.com/imperial-genomics-facility/igf-dockerfiles/blob/main/LICENSE)


* **Project name:** {{ COSMX_PROJECT_NAME }}
* **Panel type:** {{ PANEL_NAME }}
* **COSMX slide name:** {{ COSMX_SLIDE_NAME }}

## Intro
Field of View (FOV) quality control is crucial in CosMx spatial transcriptomics experiments because technical effects can significantly compromise data integrity and lead to misleading biological interpretations. While most FOVs typically perform comparably, some may experience reduced overall gene expression or selective signal loss from specific genes, potentially causing cells within an affected FOV to artificially cluster as the same cell type due to technical artifacts rather than true biological similarity. These quality issues can stem from various factors including poor tissue or section quality, high autofluorescence, and inadequate fiducials, making early detection and exclusion of problematic FOVs essential for reliable downstream analyses.

The FOV QC process involves a systematic two-step approach implemented through specialized R functions. 
* First, a permissive signal strength assessment identifies FOVs with greater than 60% signal loss across most of their spatial span by comparing total counts in grid squares to similar regions in other FOVs.
* Second, a more sophisticated bias detection method examines gene expression profiles at the reporter level rather than the gene level, since FOV quality issues are often linked to specific fluorescent reporters.


## Source
Bruker blog post: [FOV QC from single-cell gene expression in spatial dataset](https://nanostring-biostats.github.io/CosMx-Analysis-Scratch-Space/posts/fov-qc/)

## Technical notes from the blog post
```
Technical details:

We place a 7x7 grid across each FOV. For each grid square, we find the 10 most similar squares in other FOVs, with “similar” being based on the square’s expression profile (we also only accept one neighbor per other FOV).

Then we score FOVs for signal loss. For each square, we compare its total counts to its comparator squares. For each reporter bit, this gives us 49 contrasts. If most (75%) of an FOV’s squares have low total counts compared to comparators, we flag the FOV.

To score FOVs for bias, we use a similar approach. For each reporter, we take the genes using the bit, and we contrast their expression in the square vs. in the average of the 10 most similar squares elsewhere. When an FOV’s grid squares consistently underexpress the relevant gene set, we flag the FOV.
```

## Optimisations (v0.0.2)
This notebook targets both **memory** and **speed** bottlenecks that cause failures on large panels (6K, WTX) even with 64 GB RAM.

### Memory fixes
1. **Chunked sparse loading** — reads the dense CSV in small chunks (10K rows), converts each chunk to sparse immediately, and discards the dense chunk before reading the next. The full dense matrix is *never* materialised. For 500K cells × 6K genes this avoids a ~24 GB dense allocation.
2. **Eliminated duplicate objects** — the original kept `atomxdata$counts`, standalone `counts`, `negcounts`, `falsecounts` simultaneously. Now we extract what we need and `rm()` intermediates aggressively.
3. **`cellxgene2squarexbit()`** — uses sparse `gene2bitmap` and avoids the intermediate `gridxgenecounts` dense conversion for 6K gene matrices.
4. **KNN query size** — reduced from `k = n_neighbors × 50` to `k = n_neighbors × 15`, cutting the FNN distance matrix by ~70%.

### Speed fixes
1. **`summarizeFOVBias()`** — replaced per-bit `lm()` with vectorised `rowsum()` + analytic t-tests (~50–100× faster).
2. **`yhat` computation** — replaced row-by-row `sapply` with sparse matrix multiply.
3. **`getNearestNeighborsByFOV()`** — `vapply` instead of `sapply`.
4. **Progress messages** and wall-clock timing throughout.


In [ ]:
INPUT_PATH <- "{{ SLIDE_FLAT_FILE_DIR }}"

slidenames <- c("{{ COSMX_SLIDE_NAME }}")

In [ ]:
## source the necessary functions (original upstream):
source("https://raw.githubusercontent.com/Nanostring-Biostats/CosMx-Analysis-Scratch-Space/Main/_code/FOV%20QC/FOV%20QC%20utils.R")
## load barcodes:
allbarcodes <- readRDS(url("https://github.com/Nanostring-Biostats/CosMx-Analysis-Scratch-Space/raw/Main/_code/FOV%20QC/barcodes_by_panel.RDS"))

In [ ]:
barcode_hash <- list(
    "(1.0) (1.0) Human RNA Immuno-oncology" = "Hs_IO",
    "(1.0) (1.0) Human RNA Universal Cell Characterization" = "Hs_UCC",
    "(1.0) (1.0) Human RNA Universal Cell Characterization; (1.0) (1.0) Human RNA Universal Cell Characterization Add-On" = "Hs_UCC",
    "(1.1) (1.1) Human RNA 6k Discovery" = "Hs_6k",
    "(1.0) (1.0) Human RNA 6k Discovery" = "Hs_6k",
    "(1.0) (1.0) Mouse RNA Neuroscience" = "Mm_Neuro",
    "(1.0) (1.0) Mouse RNA Neuroscience; (1.0) (1.0) Mouse RNA Neuroscience Add-On" = "Mm_Neuro",
    "(1.0) (1.0) Mouse RNA Neuroscience; (1.1) (1.1) Mouse RNA Neuroscience Add-On" = "Mm_Neuro",
    "(1.0) (1.0) Mouse RNA Universal Cell Characterization" = "Mm_UCC",
    "(1.0) (1.0) Human RNA Whole Transcriptome" = "Hs_WTX"
)

In [ ]:
panel_name <- "{{ PANEL_NAME }}"

In [ ]:
panel_short_code <- barcode_hash[[panel_name]]
if (is.null(panel_short_code)) {
    stop(paste("Unknown slide panel found", ": ",panel_name))
}

In [ ]:
barcodemap <- allbarcodes[[panel_short_code]]

### Override bottleneck functions
The cell below redefines the slow/memory-hungry functions. The originals loaded above are overwritten.

In [ ]:
## ============================================================
## OPTIMISED FUNCTION OVERRIDES (memory + speed)
## ============================================================

## ---------- 1. summarizeFOVBias ----------
## ORIGINAL: runs lm(resid[,bit] ~ factor(gridfov)) for EVERY bit
##   For 6K panel: ~100 bits x full lm() = hours of runtime
## OPTIMISED: rowsum() + analytic t-test, all bits at once
summarizeFOVBias <- function(resid, gridfov, max_prop_loss) {
  gridfov <- gridfov[rownames(resid)]
  fovs <- unique(gridfov)
  nfovs <- length(fovs)
  
  fov_factor <- factor(gridfov, levels = fovs)
  fov_counts <- tabulate(fov_factor, nbins = nfovs)
  names(fov_counts) <- fovs
  
  # FOV-level means via rowsum (vectorised across all bits)
  fov_sums <- rowsum(resid, group = fov_factor, reorder = FALSE)
  fov_means <- fov_sums / fov_counts
  
  # Within-group variance
  fov_sumsq <- rowsum(resid^2, group = fov_factor, reorder = FALSE)
  fov_varsums <- fov_sumsq - fov_sums^2 / fov_counts
  
  n_total <- nrow(resid)
  ss_within <- colSums(fov_varsums)
  df_within <- n_total - nfovs
  mse <- ss_within / df_within
  
  bias <- fov_means
  se_mat <- sqrt(outer(1/fov_counts, mse))
  t_stats <- bias / se_mat
  p <- 2 * pt(-abs(t_stats), df = df_within)
  
  # Proportion of squares below threshold
  threshold <- log2(1 - max_prop_loss) / 3
  below_thresh <- (resid < threshold) * 1
  propagree_sums <- rowsum(below_thresh, group = fov_factor, reorder = FALSE)
  propagree <- propagree_sums / fov_counts
  
  rownames(bias) <- rownames(p) <- rownames(propagree) <- fovs
  colnames(bias) <- colnames(p) <- colnames(propagree) <- colnames(resid)
  
  flag <- (bias < log2(1 - max_prop_loss)) * (p < 0.01) * (propagree >= 0.5)
  return(list(flag = flag, bias = bias, p = p, propagree = propagree))
}


## ---------- 2. getNearestNeighborsByFOV ----------
## MEMORY: reduced k multiplier 50 -> 15 (saves ~70% of FNN distance matrix)
## SPEED: vapply instead of sapply
getNearestNeighborsByFOV <- function(x, gridfov, n_neighbors = 10) {
  gridfov <- gridfov[rownames(x)]
  k_query <- min(n_neighbors * 15, nrow(x))
  message(paste("  KNN search: k =", k_query, "for", nrow(x), "grid squares"))
  
  topneighbors <- FNN::get.knnx(data = x, query = x, k = k_query)$nn.index
  
  gridfov_vec <- as.character(gridfov)
  reducedneighbors <- vapply(1:nrow(topneighbors), function(i) {
    nn <- topneighbors[i, ]
    nn_fovs <- gridfov_vec[nn]
    my_fov <- gridfov_vec[i]
    keep <- (nn_fovs != my_fov) & !duplicated(nn_fovs)
    valid <- nn[keep]
    if (length(valid) >= n_neighbors) {
      valid[1:n_neighbors]
    } else {
      c(valid, rep(NA_integer_, n_neighbors - length(valid)))
    }
  }, integer(n_neighbors))
  
  rm(topneighbors); gc(verbose = FALSE)
  return(t(reducedneighbors))
}


## ---------- 3. cellxgene2squarexbit ----------
## MEMORY: use sparse gene2bitmap, avoid dense intermediate for large gene panels
cellxgene2squarexbit <- function(counts, grid, genes, barcodes) {
  colorvals <- getcolorvals(barcodes)
  nreportercycles <- nchar(barcodes[1]) / 2
  nbits <- nreportercycles * 4
  
  inasquare <- !is.na(grid)
  counts <- counts[inasquare, ]
  grid <- grid[inasquare]
  
  grid_factor <- as.factor(grid)
  gridmap <- Matrix::sparseMatrix(
    i = as.numeric(grid_factor),
    j = seq_len(length(grid)),
    x = 1,
    dims = c(nlevels(grid_factor), length(grid)))
  rownames(gridmap) <- levels(grid_factor)
  
  # Grid x gene mean expression (stays sparse)
  ncellspergrid <- Matrix::rowSums(gridmap)
  gridxgenecounts <- Matrix::Diagonal(x = 1/ncellspergrid) %*% (gridmap %*% counts)
  rownames(gridxgenecounts) <- levels(grid_factor)
  
  # Build SPARSE gene-to-bit mapping matrix
  gene2bitmap <- barcode2bitmatrix_sparse(barcodes)
  rownames(gene2bitmap) <- genes
  
  sharedgenes <- intersect(genes, colnames(counts))
  gridxbitcounts <- gridxgenecounts[, sharedgenes] %*% gene2bitmap[sharedgenes, ]
  
  # Only convert the small bit-level matrix to dense
  return(as.matrix(gridxbitcounts))
}

## Sparse version of barcode2bitmatrix
barcode2bitmatrix_sparse <- function(barcodes) {
  colorvals <- getcolorvals(barcodes)
  nreportercycles <- nchar(barcodes[1]) / 2
  nbits <- nreportercycles * 4
  bitnames <- paste0("reportercycle", rep(seq_len(nreportercycles), each = 4), rep(colorvals, nreportercycles))
  
  # Build as triplets
  ii <- c(); jj <- c()
  for (cyc in seq_len(nreportercycles)) {
    barcodeposition <- cyc * 2
    barcodehere <- substr(barcodes, barcodeposition, barcodeposition)
    for (ci in seq_along(colorvals)) {
      col <- colorvals[ci]
      gene_idx <- which(barcodehere == col)
      bit_idx <- (cyc - 1) * 4 + ci
      ii <- c(ii, gene_idx)
      jj <- c(jj, rep(bit_idx, length(gene_idx)))
    }
  }
  mat <- Matrix::sparseMatrix(i = ii, j = jj, x = 1,
                              dims = c(length(barcodes), nbits),
                              dimnames = list(NULL, bitnames))
  return(mat)
}


## ---------- 4. runFOVQC ----------
## Vectorised yhat via sparse matrix multiply instead of row-by-row sapply
## Aggressive gc() between stages
runFOVQC <- function(counts, xy, fov, tissue = NULL, barcodemap, max_prop_loss = 0.6, max_totalcounts_loss = 0.6) {
  
  if ((max_prop_loss > 1) | (max_prop_loss < 0)) {
    stop("max_prop_loss must fall in range of 0-1.")
  }
  fov <- paste0(tissue, as.character(fov))
  
  mem_report <- function(stage) {
    gc_info <- gc(verbose = FALSE)
    used_mb <- sum(gc_info[, "used"]) / 1024
    message(paste0("  [Memory] After ", stage, ": ~", round(used_mb, 0), " MB used"))
  }
  
  message("Step 1/5: Building FOV grids...")
  gridinfo <- makeGrid(xy = xy, fov = fov, squares_per_fov = 49, min_cells_per_square = 10)
  mem_report("grid construction")
  
  message("Step 2/5: Converting cell×gene to grid×bit...")
  bitcounts <- cellxgene2squarexbit(counts = counts,
                                    grid = gridinfo$gridid,
                                    genes = barcodemap$gene,
                                    barcodes = barcodemap$barcode)
  gc(verbose = FALSE)
  mem_report("grid×bit conversion")
  
  bitcounts <- sweep(bitcounts, 1, rowSums(bitcounts), "/") * mean(rowSums(bitcounts))
  
  message("Step 3/5: Finding nearest neighbours across FOVs...")
  comparators <- getNearestNeighborsByFOV(x = bitcounts,
                                          gridfov = gridinfo$gridfov,
                                          n_neighbors = 10)
  mem_report("KNN")
  
  ## Build sparse comparator weight matrix (used for both totalcounts and yhat)
  n_sq <- nrow(comparators)
  n_nb <- ncol(comparators)
  valid <- !is.na(comparators)
  row_idx <- rep(1:n_sq, n_nb)[as.vector(valid)]
  col_idx <- as.vector(comparators)[as.vector(valid)]
  counts_per_row <- rowSums(valid)
  weights <- rep(1 / pmax(counts_per_row, 1), n_nb)[as.vector(valid)]
  comp_mat <- Matrix::sparseMatrix(i = row_idx, j = col_idx, x = weights,
                                   dims = c(n_sq, n_sq))
  
  ## --- Total counts outlier detection ---
  totalcounts <- Matrix::rowSums(counts)
  totalcountsgrid <- by(totalcounts, gridinfo$gridid, mean)[rownames(bitcounts)]
  totalcountshat <- as.vector(comp_mat %*% as.vector(totalcountsgrid))
  totalcountsresids <- log2(as.vector(totalcountsgrid) / totalcountshat)
  names(totalcountsresids) <- names(totalcountsgrid)
  
  flaggedgrids <- totalcountsresids < log2(1 - max_totalcounts_loss)
  flaggedgridsperfov <- by(as.vector(1*flaggedgrids), gridinfo$gridfov[names(flaggedgrids)], mean)
  flaggedfovs_fortotalcounts <- names(which(flaggedgridsperfov > 0.75))
  
  ## --- Bias detection ---
  message("Step 4/5: Computing expected expression (vectorised)...")
  yhat <- as.matrix(comp_mat %*% bitcounts)
  rownames(yhat) <- rownames(bitcounts)
  rm(comp_mat); gc(verbose = FALSE)
  
  resid <- log2((bitcounts + 1) / (yhat + 1))
  rownames(resid) <- rownames(bitcounts)
  rm(yhat); gc(verbose = FALSE)
  mem_report("residual computation")
  
  message("Step 5/5: Summarising FOV bias (vectorised)...")
  fovstats <- summarizeFOVBias(resid = resid, gridfov = gridinfo$gridfov, max_prop_loss = max_prop_loss)
  
  flags_per_fov_x_reportercycle <- c()
  reportercycle <- substr(colnames(fovstats$flag), 1, nchar(colnames(fovstats$flag)) - 1)
  for (rc in unique(reportercycle)) {
    flags_per_fov_x_reportercycle <- cbind(flags_per_fov_x_reportercycle,
                                           rowMeans(fovstats$flag[, reportercycle == rc, drop = FALSE]))
  }
  colnames(flags_per_fov_x_reportercycle) <- unique(reportercycle)
  
  flaggedfovs_forbias <- rownames(flags_per_fov_x_reportercycle)[rowSums(flags_per_fov_x_reportercycle >= 0.5) > 0]
  flaggedfovs <- union(flaggedfovs_fortotalcounts, flaggedfovs_forbias)
  
  if (length(flaggedfovs) > 0) {
    message(paste0("The following FOVs failed QC for one or more barcode positions: ",
                   paste0(flaggedfovs, collapse = ", ")))
  } else {
    message("All FOVs passed QC")
  }
  
  flagged_fov_x_gene <- c()
  for (f in flaggedfovs) {
    flaggedreportercycles <- names(which(flags_per_fov_x_reportercycle[f, ] >= 0.5))
    for (rc in flaggedreportercycles) {
      reporterposition <- as.numeric(substr(rc, 14, nchar(rc) - 1)) * 2
      flaggedgenes <- barcodemap$gene[substr(barcodemap$barcode, reporterposition, reporterposition) != "."]
      tempflags <- cbind(rep(f, length(flaggedgenes)), flaggedgenes)
      colnames(tempflags) <- c("fov", "gene")
      flagged_fov_x_gene <- rbind(flagged_fov_x_gene, tempflags)
    }
  }
  
  mem_report("final")
  
  return(list(flaggedfovs = flaggedfovs,
              flaggedfovs_fortotalcounts = flaggedfovs_fortotalcounts,
              flaggedfovs_forbias = flaggedfovs_forbias,
              flagged_fov_x_gene = flagged_fov_x_gene,
              flags_per_fov_x_reportercycle = flags_per_fov_x_reportercycle,
              fovstats = fovstats, resid = resid, totalcountsresids = totalcountsresids,
              gridinfo = gridinfo, xy = xy, fov = fov))
}

message("Optimised function overrides loaded.")


### Data loading (memory-safe chunked sparse)
The key memory fix: reads the dense CSV in small chunks (configurable, default 10K rows), converts each chunk to a sparse matrix immediately, and discards the dense chunk. The full dense matrix is **never** held in memory.

For 500K cells × 6K genes:
- **Original approach**: `fread()` loads ~24 GB dense → `as(, 'sparseMatrix')` needs another ~24 GB temporary = **~48 GB peak**
- **This approach**: holds only one 10K-row chunk dense (~0.5 GB) + growing sparse matrix (~2-4 GB) = **~5 GB peak**

In [ ]:
# necessary libraries
library(data.table)
library(Matrix)
library(viridis)
library(uwot)
library(irlba)

setDTthreads(0L)
message(paste("data.table using", getDTthreads(), "threads"))

## ============================================================
## MEMORY-SAFE CHUNKED SPARSE LOADING
## ============================================================
CHUNK_SIZE <- 10000L  # rows per chunk - tune based on available RAM
                       # 10K rows x 6K genes x 8 bytes = ~480 MB per dense chunk
                       # Reduce to 5000 if still running out of memory

countlist <- vector(mode="list", length = length(slidenames))
metadatalist <- vector(mode="list", length = length(slidenames))

for (i in 1:length(slidenames)) {
  slidename <- slidenames[i]
  message(paste0("\n=== Loading slide ", slidename, " (", i, "/", length(slidenames), ") ==="))
  
  thisslidesfiles <- dir(paste0(INPUT_PATH, "/", slidename))
  
  # Load metadata (small)
  thisslidesmetadata <- thisslidesfiles[grepl("metadata\\_file", thisslidesfiles)]
  tempdatatable <- data.table::fread(paste0(INPUT_PATH, "/", slidename, "/", thisslidesmetadata))
  slide_ID_numeric <- tempdatatable[1, ]$slide_ID
  
  # Identify counts file
  thisslidescounts <- thisslidesfiles[grepl("exprMat\\_file", thisslidesfiles)]
  countsfile <- paste0(INPUT_PATH, "/", slidename, "/", thisslidescounts)
  
  # Read header only to get column names and count rows
  header_dt <- fread(countsfile, nrows = 1)
  header <- colnames(header_dt)
  gene_cols <- setdiff(header, c("fov", "cell_ID"))
  n_gene_cols <- length(gene_cols)
  rm(header_dt)
  
  # Count total rows (read only 2 small columns)
  id_cols <- fread(countsfile, select = c("fov", "cell_ID"))
  number_of_cells <- nrow(id_cols)
  rm(id_cols)
  gc(verbose = FALSE)
  
  message(paste0("  ", number_of_cells, " cells x ", n_gene_cols, " genes"))
  message(paste0("  Loading in chunks of ", CHUNK_SIZE, " rows..."))
  
  # Chunked reading: read dense chunk -> convert to sparse -> discard dense
  number_of_chunks <- ceiling(number_of_cells / CHUNK_SIZE)
  sparse_chunks <- vector(mode = "list", length = number_of_chunks)
  rowname_chunks <- vector(mode = "list", length = number_of_chunks)
  
  skiprows <- 0
  for (ch in 1:number_of_chunks) {
    # How many rows to read in this chunk
    nrows_this <- min(CHUNK_SIZE, number_of_cells - skiprows)
    
    chunk_dt <- data.table::fread(
      countsfile,
      nrows = nrows_this,
      skip = skiprows + (ch > 1),  # +1 to skip header after first chunk
      header = (ch == 1)
    )
    if (ch > 1) colnames(chunk_dt) <- header
    
    # Build row names
    rn <- paste0("c_", slide_ID_numeric, "_", chunk_dt$fov, "_", chunk_dt$cell_ID)
    rowname_chunks[[ch]] <- rn
    
    # Convert gene columns to sparse IMMEDIATELY, discard dense
    chunk_mat <- as.matrix(chunk_dt[, ..gene_cols])
    sparse_chunks[[ch]] <- as(chunk_mat, "dgCMatrix")
    rownames(sparse_chunks[[ch]]) <- rn
    
    rm(chunk_dt, chunk_mat, rn)
    if (ch %% 10 == 0 || ch == number_of_chunks) {
      gc(verbose = FALSE)
      message(paste0("    chunk ", ch, "/", number_of_chunks, " done"))
    }
    
    skiprows <- skiprows + nrows_this
  }
  
  # Combine sparse chunks
  message("  Combining sparse chunks...")
  countlist[[i]] <- do.call(rbind, sparse_chunks)
  rm(sparse_chunks, rowname_chunks)
  gc(verbose = FALSE)
  
  # Reorder to match metadata
  slide_fov_cell_metadata <- paste0("c_", slide_ID_numeric, "_", tempdatatable$fov, "_", tempdatatable$cell_ID)
  countlist[[i]] <- countlist[[i]][match(slide_fov_cell_metadata, rownames(countlist[[i]])), ]
  metadatalist[[i]] <- tempdatatable
  
  if (i == 1) {
    sharedgenes <- colnames(countlist[[i]])
    sharedcolumns <- colnames(tempdatatable)
  } else {
    sharedgenes <- intersect(sharedgenes, colnames(countlist[[i]]))
    sharedcolumns <- intersect(sharedcolumns, colnames(tempdatatable))
  }
  
  rm(tempdatatable)
  gc(verbose = FALSE)
}

# Reduce to shared columns/genes
for (i in 1:length(slidenames)) {
  metadatalist[[i]] <- metadatalist[[i]][, ..sharedcolumns]
  countlist[[i]] <- countlist[[i]][, sharedgenes]
}
counts <- do.call(rbind, countlist)
metadata <- rbindlist(metadatalist)

# FREE intermediate lists immediately
rm(countlist, metadatalist)
gc(verbose = FALSE)

# FOV IDs
metadata$FOV <- paste0("FOV_", metadata$fov)
metadata$cell_ID <- NULL

# Separate controls — but DON'T keep originals around
neg_idx <- grepl("Negative", colnames(counts))
false_idx <- grepl("SystemControl", colnames(counts))
gene_idx <- !neg_idx & !false_idx

negcounts <- counts[, neg_idx]
falsecounts <- counts[, false_idx]
counts <- counts[, gene_idx]

# DO NOT create atomxdata list — it duplicates everything
# Just use counts, metadata, negcounts, falsecounts directly

xy <- as.matrix(metadata[, c("CenterX_global_px", "CenterY_global_px")])
rownames(xy) <- metadata$cell_id
thisinstrument_nanometers_per_pixel <- 120.280945
xy <- xy * thisinstrument_nanometers_per_pixel / 1000000
colnames(xy) <- paste0(c("x", "y"), "_mm")

count_threshold <- 20
flag <- metadata$nCount_RNA < count_threshold

gc_info <- gc(verbose = FALSE)
used_mb <- sum(gc_info[, "used"]) / 1024
message(paste0("\nData loaded: ", nrow(counts), " cells x ", ncol(counts), " genes"))
message(paste0("Current R memory usage: ~", round(used_mb, 0), " MB"))
message(paste0("Sparse counts matrix: ", format(object.size(counts), units = "GB")))


This table shows the count of each unique value in the flag table.

In [ ]:
table(flag)

### Cell area distribution
Following histogram plot shows distribution of cell areas in this dataset.

* Peak of the distribution occurs shows the most common cell area.
* The spread shows the minimum and maximum cell sizes detected.
* Normal distribution suggests consistent cell segmentation.
* Right-skewed distribution (long tail toward larger areas) is common in biological data.
* Multiple peaks might indicate different cell types or segmentation artifacts.
* Very large or very small areas that might represent:
  * Segmentation errors (cells that are too small or artificially merged)
  * Debris or artifacts
  * Genuinely different cell types

This histogram is often used to identify thresholds for filtering out poorly segmented cells. For example, you might exclude cells that are extremely small (likely debris) or extremely large (likely merged cells or artifacts). The distribution helps you choose reasonable cutoff values for downstream analysis.

In [ ]:
## set plot size
options(repr.plot.width=15, repr.plot.height=10)
# what's the distribution of areas?
hist(metadata$Area, breaks = 100, xlab = "Cell Area", main = "")
# based on the above, set a threshold:
area_threshold <- 30000
abline(v = area_threshold, col = "red")

### Number of flagged cells based on area

In [ ]:
# flag cells based on area:
flag <- flag | (metadata$Area > area_threshold)
table(flag)

### List of QC failed FOVs

In [ ]:
## Run FOV QC (with timing + memory monitoring)
message("Starting FOV QC analysis...")
message(paste0("Input: ", nrow(counts), " cells x ", ncol(counts), " genes"))
start_time <- Sys.time()

fovqcresult <- runFOVQC(counts = counts, xy = xy, fov = metadata$FOV,
                        barcodemap = barcodemap, max_prop_loss = 0.6)

end_time <- Sys.time()
elapsed <- difftime(end_time, start_time, units = "mins")
message(paste0("\nFOV QC completed in ", round(elapsed, 2), " minutes"))

# Free the large counts matrix if no longer needed
# (uncomment if you don't need it for downstream analysis)
# rm(counts); gc()


### Flagged FOVs

This visualization creates a spatial plot showing which FOVs (Fields of View) have been flagged as problematic during the quality control process. It helps identify whether problematic FOVs are clustered together (suggesting localized technical issues like poor tissue preparation) or randomly distributed. Colors passing FOVs in <span style="color:steelblue">blue</span> and failing FOVs are in <span style="color:red">red</span>.

In [ ]:
## set plot size
options(repr.plot.width=18, repr.plot.height=12)
# map of flagged FOVs:
mapFlaggedFOVs(fovqcresult, shownames = TRUE)

#### List FOVs flagged for any reason, for loss of signal, for bias

In [ ]:
# list FOVs flagged for any reason, for loss of signal, for bias:
fovqcresult$flaggedfovs

#### List FOVs flagged for loss of signal

In [ ]:
fovqcresult$flaggedfovs_fortotalcounts

#### List FOVs flagged for bias
This identifies FOVs where certain barcode bits (fluorescent reporters) are systematically under-expressed compared to biologically similar regions in other FOVs.

* The algorithm examines each of the 4 colors (B, G, Y, R) within each reporter cycle.
* For each FOV's 7×7 grid squares, it compares expression of genes using specific barcode bits to the 10 most similar grid squares from other FOVs
* An FOV gets flagged for bias if:
  * At least 2 out of 4 colors from the same reporter cycle show significant bias (≥50% of that reporter cycle's bits are flagged)
  * The bias exceeds the max_prop_loss threshold (default 60% signal loss)
  * The effect is statistically significant (p < 0.01)
  * At least 50% of the FOV's grid squares agree on the direction of the bias

In [ ]:
fovqcresult$flaggedfovs_forbias

### FOV signal loss pattern

Following spatial visualization shows signal loss patterns across FOVs by displaying how total gene expression counts in each region compare to similar regions elsewhere. It helps researchers identify FOVs where cells are systematically expressing fewer total transcripts than expected based on comparable spatial regions in other FOVs. The yellow borders make it immediately clear which FOVs have been flagged for having >60% signal loss. <span style="color:#FFD700">Yellow</span> rectangles with <b>thick borders</b> highlight FOVs that failed the total counts QC check. 

Each cell is plotted as a small dot colored according to its log2 fold-change in total counts compared to similar spatial regions:

* Dark blue: Severe signal loss (< -2 fold-change)
* Blue: Moderate signal loss (-1 fold-change)
* Grey: Normal signal (0 fold-change)
* Red: Higher signal (+1 fold-change)
* Dark red: Much higher signal (> +2 fold-change)

In [ ]:
FOVSignalLossSpatialPlot(fovqcresult, shownames = TRUE) 

### Spatial plots of flagged reporter positions
This plot shows how specific barcode bits (fluorescent reporters) are performing across different spatial regions within FOVs, helping identify technical quality issues. It helps to validate FOV QC results by showing whether apparent expression differences are due to technical FOV effects or genuine biological spatial patterns.

Color Coding:

* Blue colors indicate underexpression (negative fold-change)
* Red colors indicate overexpression (positive fold-change)
* Gray indicates no change
* The scale typically ranges from -1 to +1 log2 fold-change

In [ ]:
# spatial plots of flagged reporter positions:
par(mfrow = c(2,2))
FOVEffectsSpatialPlots(res = fovqcresult, outdir = NULL, bits = "flagged_reportercycles") 

## FOV bias
This heatmap that shows the bias in barcode bit expression across all FOVs and reporter bits.

* Rows: Each FOV in the experiment
* Columns: Each barcode bit (reporter cycle + color combination, e.g., "reportercycle1B", "reportercycle1G", etc.)
* Cell Values: Log2 fold-change showing how much each FOV under- or over-expresses genes using specific barcode bits compared to similar regions in other FOVs

Color Scheme:

* Dark blue: Severe underexpression (< -2 fold-change)
* Blue: Moderate underexpression
* White: Normal expression (0 fold-change)
* Red: Moderate overexpression
* Dark red: Severe overexpression (> +2 fold-change)

The heatmap makes it easy to spot whether failures are:

* Clustered within specific reporter cycles (technical issue)
* Scattered randomly (potentially biological variation)
* Affecting multiple FOVs in similar patterns

In [ ]:
## set plot size
options(repr.plot.width=25, repr.plot.height=18)
FOVEffectsHeatmap(fovqcresult)

### List of affected genes
A list of all the unique gene names whose expression is compromised in one or more flagged FOVs.

In [ ]:
# genes from all barcode bits that were flagged in any FOV:
print(unique(fovqcresult$flagged_fov_x_gene[,"gene"]))

<!-- ## Memory troubleshooting
If you still run out of memory, try these in order:

1. **Reduce `CHUNK_SIZE`** in the data loading cell from 10000 to 5000 or even 2000
2. **Free `counts` after `runFOVQC`**: add `rm(counts, negcounts, falsecounts); gc()` after the QC call — the results object holds everything needed for plotting
3. **Reduce KNN multiplier**: in the `getNearestNeighborsByFOV` override, change `n_neighbors * 15` to `n_neighbors * 10` (minimum safe value)
4. **Subsample cells**: if >500K cells, consider randomly sampling 80% before QC — FOV-level QC is robust to subsampling
5. **Check `object.size`**: run `sort(sapply(ls(), function(x) object.size(get(x))))` to find the largest objects in your R session -->